# Project 1: J.Lindeberg product scraper (shop.standrews.com)

**Flow**
1. Check robots.txt and record the result
2. Open homepage, accept cookies
3. Open Brands menu, go to J.Lindeberg (verified, with fallback)
4. Collect product links (with optional "load more" pagination)
5. Loop over product pages, extract fields, save to CSV
6. Write a collection log (date, observations, restrictions)

Also read the site's terms of service manually and record what you find in your write-up.

In [1]:
import time
import json
from datetime import datetime
from urllib import robotparser

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

## Settings (document these in your write-up)

In [2]:
BASE_URL = "https://shop.standrews.com"
STARTING_URL = BASE_URL + "/homepage/"
BRAND_NAME = "J.Lindeberg"
MAX_PRODUCTS = 50          # restriction placed on collection
DELAY_SECONDS = 2          # polite delay between product requests
MAX_LOAD_MORE_CLICKS = 5   # pagination limit
OUTPUT_CSV = "jlindeberg_products.csv"
LOG_JSON = "collection_log.json"

PRODUCT_LINK_XPATH = "//a[contains(@href, '/catalog/products/')]"

## Helper functions

In [3]:
def check_robots(url_to_check):
    """Return (allowed, note). Also read the site's terms of service manually."""
    rp = robotparser.RobotFileParser()
    rp.set_url(BASE_URL + "/robots.txt")
    try:
        rp.read()
        allowed = rp.can_fetch("*", url_to_check)
        return allowed, f"robots.txt read; can_fetch('*', {url_to_check}) = {allowed}"
    except Exception as e:
        return None, f"Could not read robots.txt ({type(e).__name__}); check manually"


def make_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1400,1000")
    return webdriver.Chrome(options=options)


def accept_cookies(driver, timeout=8):
    """Click 'Accept all' if the banner appears."""
    try:
        btn = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[normalize-space()='Accept all']")
            )
        )
        btn.click()
        print("Cookie banner accepted")
    except TimeoutException:
        print("Cookie banner not found, continuing")

In [4]:
def open_brand(driver, brand_name):
    """
    Open the Brands menu and navigate to the brand page.
    Verifies which link matched, clicks it with JavaScript (avoids
    coordinate misclicks on neighbours like Peter Millar), and falls back
    to driver.get(href) if the click lands on the wrong page.
    """
    wait = WebDriverWait(driver, 10)

    # Open the Brands menu. Try the known-working absolute XPath first,
    # then text-based fallbacks in case the layout shifts.
    lower = "translate(normalize-space(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz')"
    brands_candidates = [
        '//*[@id="root"]/div/div[1]/div[1]/div/div[2]/div[3]/nav/div/ul/li[6]/span',
        "//nav//li/span[normalize-space()='Brands']",
        f"//nav//*[self::span or self::a or self::button][{lower}='brands']",
        f"//*[self::span or self::a or self::button][contains({lower}, 'brand')]",
    ]
    brands_menu = None
    for xp in brands_candidates:
        try:
            brands_menu = WebDriverWait(driver, 6).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            print("Brands menu found with:", xp)
            break
        except TimeoutException:
            print("No match for:", xp)
    if brands_menu is None:
        raise TimeoutException(
            "Brands menu not found. Check for an overlay (cookie banner) "
            "or inspect the menu element and add its XPath to brands_candidates."
        )
    brands_menu.click()

    # Find every anchor for this brand and report what we see
    anchor_xpath = f"//a[@title='{brand_name}']"
    wait.until(EC.presence_of_element_located((By.XPATH, anchor_xpath)))
    matches = driver.find_elements(By.XPATH, anchor_xpath)

    print(f"Found {len(matches)} link(s) titled '{brand_name}':")
    for m in matches:
        print("  ", m.get_attribute("href"), "| displayed:", m.is_displayed())

    # Prefer a visible match whose href mentions the brand
    key = brand_name.lower().replace(".", "")
    target = None
    for m in matches:
        href = (m.get_attribute("href") or "").lower().replace(".", "")
        if m.is_displayed() and key in href:
            target = m
            break
    if target is None:
        target = matches[0]

    brand_url = target.get_attribute("href")
    print("Target brand URL:", brand_url)

    # One genuine click (JS click hits this exact element)
    try:
        driver.execute_script("arguments[0].click();", target)
        time.sleep(2)
    except ElementClickInterceptedException:
        pass

    # Verify we landed on the right page; otherwise navigate directly
    if key not in driver.current_url.lower().replace(".", ""):
        print("Click landed on", driver.current_url, "- navigating directly")
        driver.get(brand_url)

    print("Page title:", driver.title)
    print("Current URL:", driver.current_url)

In [5]:
def collect_product_links(driver, max_products):
    """Collect unique product URLs, clicking 'load more' / 'next' if present."""
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.XPATH, PRODUCT_LINK_XPATH)))

    links = []
    clicks = 0
    while True:
        for a in driver.find_elements(By.XPATH, PRODUCT_LINK_XPATH):
            href = a.get_attribute("href")
            if href and href not in links:
                links.append(href)
        print("Unique product links so far:", len(links))

        if len(links) >= max_products or clicks >= MAX_LOAD_MORE_CLICKS:
            break

        # Try a pagination control (adjust the text after inspecting the page)
        more = driver.find_elements(
            By.XPATH,
            "//button[contains(translate(normalize-space(), "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), 'load more')"
            " or contains(translate(normalize-space(), "
            "'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), 'show more')]"
            " | //a[@rel='next']",
        )
        more = [b for b in more if b.is_displayed()]
        if not more:
            break

        before = len(links)
        driver.execute_script("arguments[0].click();", more[0])
        clicks += 1
        time.sleep(DELAY_SECONDS)
        # stop if nothing new appeared
        now = {
            a.get_attribute("href")
            for a in driver.find_elements(By.XPATH, PRODUCT_LINK_XPATH)
        }
        if len(now - set(links)) == 0 and len(links) == before:
            break

    return links[:max_products]


def first_text(driver, xpaths):
    """Return text of the first XPath that matches, else None."""
    for xp in xpaths:
        try:
            el = driver.find_element(By.XPATH, xp)
            text = el.text.strip()
            if text:
                return text
        except NoSuchElementException:
            continue
    return None


def scrape_product(driver, url):
    """Extract fields from one product page. Adjust XPaths after inspecting."""
    driver.get(url)
    wait = WebDriverWait(driver, 10)
    name = wait.until(EC.presence_of_element_located((By.XPATH, "//h1"))).text.strip()

    price = first_text(driver, [
        "//*[contains(@class,'rice')]",
        "//*[contains(text(),'$')]",
    ])
    rating = first_text(driver, [
        "//*[contains(@class,'ating')]",
        "//*[contains(@aria-label,'star')]",
    ])
    description = first_text(driver, [
        "//*[contains(@class,'escription')]",
        "//*[@id='description']",
    ])

    # Categories from breadcrumbs, if present
    crumbs = driver.find_elements(
        By.XPATH, "//*[contains(@class,'readcrumb')]//a | //nav[@aria-label='breadcrumb']//a"
    )
    categories = " > ".join(c.text.strip() for c in crumbs if c.text.strip()) or None

    return {
        "name": name,
        "vendor": BRAND_NAME,
        "price": price,
        "rating": rating,
        "description": description,
        "categories": categories,
        "url": url,
        "scraped_at": datetime.now().isoformat(timespec="seconds"),
    }

## Run the scraper
Run these cells in order. The driver stays open between cells, so you can inspect the browser if something fails.

### Step 1: Compliance check

In [6]:
allowed, robots_note = check_robots(STARTING_URL)
print(robots_note)
if allowed is False:
    raise SystemExit("robots.txt disallows this path. Choose a different site.")

robots.txt read; can_fetch('*', https://shop.standrews.com/homepage/) = True


### Step 2: Start browser, open homepage, accept cookies

In [7]:
started = datetime.now().isoformat(timespec="seconds")
records, errors = [], []

driver = make_driver()
driver.get(STARTING_URL)
print(driver.title, "|", driver.current_url)

accept_cookies(driver)

St Andrews Links : The Home of Golf | https://shop.standrews.com/homepage/
Cookie banner accepted


### Step 3: Navigate to the J.Lindeberg brand page (automated click)

In [8]:
open_brand(driver, BRAND_NAME)

Brands menu found with: //*[@id="root"]/div/div[1]/div[1]/div/div[2]/div[3]/nav/div/ul/li[6]/span
Found 1 link(s) titled 'J.Lindeberg':
   None | displayed: True
Target brand URL: None
Page title: Shop - St Andrews Links : The Home of Golf
Current URL: https://shop.standrews.com/catalog/category/jlindeberg


### Step 4: Collect product links

In [9]:
product_links = collect_product_links(driver, MAX_PRODUCTS)
print("Number of product links to visit:", len(product_links))

Unique product links so far: 12
Number of product links to visit: 12


### Step 5: Loop over product pages and extract data

In [10]:
for i, url in enumerate(product_links, start=1):
    try:
        rec = scrape_product(driver, url)
        records.append(rec)
        print(f"[{i}/{len(product_links)}] {rec['name']} | {rec['price']}")
    except Exception as e:
        errors.append({"url": url, "error": type(e).__name__})
        print(f"[{i}/{len(product_links)}] skipped {url}: {type(e).__name__}")
    time.sleep(DELAY_SECONDS)

[1/12] J.Lindeberg Heyden Quilt Hybrid Jacket | £240.00
[2/12] J.Lindeberg Tour Tech Polo | £120.00
[3/12] J.Lindeberg Club Hoodie | £180.00
[4/12] J.Lindeberg Tour Tech Polo | £125.00
[5/12] J.Lindeberg Tour Tech Print Polo | £135.00
[6/12] J.Lindeberg Lukas 1/4 Zip Midlayer | £185.00
[7/12] J.Lindeberg Sole Hoodie | £200.00
[8/12] J.Lindeberg Heath Polo | £110.00
[9/12] J.Lindeberg Heath Polo | £120.00
[10/12] J.Lindeberg Leon 1/4 Zip Midlayer | £165.00
[11/12] J.Lindeberg Heyden Quilt Hybrid Jacket | £240.00
[12/12] J.Lindeberg Heyden Quilt Hybrid Vest | £205.00


### Step 6: Close browser, save data and collection log

In [11]:
driver.quit()

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

log = {
    "website": BASE_URL,
    "brand": BRAND_NAME,
    "collection_started": started,
    "collection_finished": datetime.now().isoformat(timespec="seconds"),
    "observations_collected": len(df),
    "pages_skipped": len(errors),
    "restrictions": {
        "max_products": MAX_PRODUCTS,
        "delay_seconds_between_requests": DELAY_SECONDS,
        "max_load_more_clicks": MAX_LOAD_MORE_CLICKS,
    },
    "robots_check": robots_note,
    "errors": errors,
}
with open(LOG_JSON, "w") as f:
    json.dump(log, f, indent=2)

print(f"Done. {len(df)} observations saved to {OUTPUT_CSV}")
print(f"Log saved to {LOG_JSON}")

Done. 12 observations saved to jlindeberg_products.csv
Log saved to collection_log.json


### Preview the data and log (use this for your write-up)

In [12]:
display(df.head())
print(json.dumps(log, indent=2))

,name,vendor,price,rating,description,categories,url,scraped_at
0,J.Lindeberg Heyden Quilt Hybrid Jacket,J.Lindeberg,£240.00,None,Every purchase helps preserve the future of th...,Home > J.Lindeberg Heyden Quilt Hybrid Jacket,https://shop.standrews.com/catalog/products/j_...,2026-10-17T20:11:57
1,J.Lindeberg Tour Tech Polo,J.Lindeberg,£120.00,None,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-17T20:12:01
2,J.Lindeberg Club Hoodie,J.Lindeberg,£180.00,None,Every purchase helps preserve the future of th...,Home > J.Lindeberg Club Hoodie,https://shop.standrews.com/catalog/products/j_...,2026-10-17T20:12:05
3,J.Lindeberg Tour Tech Polo,J.Lindeberg,£125.00,None,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-17T20:12:09
4,J.Lindeberg Tour Tech Print Polo,J.Lindeberg,£135.00,None,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Print Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-17T20:12:15


{
  "website": "https://shop.standrews.com",
  "brand": "J.Lindeberg",
  "collection_started": "2026-10-17T20:11:47",
  "collection_finished": "2026-10-17T20:12:47",
  "observations_collected": 12,
  "pages_skipped": 0,
  "restrictions": {
    "max_products": 50,
    "delay_seconds_between_requests": 2,
    "max_load_more_clicks": 5
  },
  "robots_check": "robots.txt read; can_fetch('*', https://shop.standrews.com/homepage/) = True",
  "errors": []
}
